## 1. Required Imports

In [15]:
!pip -q install -U openai
!pip install -U openai-agents python-dotenv
import os
import json
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.optim as optim
import getpass
import copy
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from openai import OpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

## 2. API Key Setup

In [16]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("OpenAI client is ready.")

OpenAI client is ready.


## 3. Load Stock Dataset

In [17]:
ds = load_dataset("Adilbai/stock-dataset")
df = ds["train"].to_pandas()

## 4. Feature Logic From GRU/LSTM Models

In [18]:
TARGET_COL = "Future_Up_1d"

leakage_cols = [
    "Future_Return_1d", "Future_Up_1d", "Future_Category_1d",
    "Future_Return_5d", "Future_Up_5d", "Future_Category_5d",
    "Future_Return_10d", "Future_Up_10d", "Future_Category_10d",
    "Future_Return_20d", "Future_Up_20d", "Future_Category_20d",
]

id_cols = ["Date", "Ticker"]

feature_cols = []
for c in df.columns:
    if c in leakage_cols or c in id_cols:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        feature_cols.append(c)

df = df.dropna(subset=feature_cols + [TARGET_COL, "Ticker", "Date"]).copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Rows:", len(df))
print("Features:", len(feature_cols))
print("Unique tickers:", df["Ticker"].nunique())

/tmp/ipykernel_9087/3891871958.py:20: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


Rows: 620095
Features: 59
Unique tickers: 503


## 5. Create Time-Based Split

In [19]:
train_cutoff = df["Date"].quantile(0.70)
val_cutoff   = df["Date"].quantile(0.85)

train_df = df[df["Date"] <= train_cutoff].copy()
val_df   = df[(df["Date"] > train_cutoff) & (df["Date"] <= val_cutoff)].copy()
test_df  = df[df["Date"] > val_cutoff].copy()

train_mean = train_df[feature_cols].mean()
train_std  = train_df[feature_cols].std().replace(0, 1.0)

train_df[feature_cols] = (train_df[feature_cols] - train_mean) / train_std
val_df[feature_cols]   = (val_df[feature_cols] - train_mean) / train_std
test_df[feature_cols]  = (test_df[feature_cols] - train_mean) / train_std

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))
print("Train cutoff:", train_cutoff)
print("Val cutoff:", val_cutoff)

Train rows: 434259
Val rows: 93284
Test rows: 92552
Train cutoff: 2024-01-04 00:00:00-05:00
Val cutoff: 2024-10-01 00:00:00-04:00


## 6. Build Sequence Windows

In [20]:
SEQ_LEN = 20

def build_examples(data: pd.DataFrame, feature_cols, target_col, seq_len=SEQ_LEN, max_per_ticker=None):
    examples = []

    for ticker, g in data.groupby("Ticker", sort=False):
        g = g.reset_index(drop=True)
        if len(g) < seq_len:
            continue

        counter = 0
        for end in range(seq_len - 1, len(g)):
            start = end - seq_len + 1
            window = g.loc[start:end].copy()
            row = g.loc[end].copy()

            examples.append({
                "Ticker": ticker,
                "Date": row["Date"],
                "target": int(row[target_col]),
                "window_df": window[["Date", "Ticker"] + feature_cols].copy()
            })

            counter += 1
            if max_per_ticker is not None and counter >= max_per_ticker:
                break

    return examples

train_examples = build_examples(train_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN, max_per_ticker=10)
val_examples   = build_examples(val_df,   feature_cols, TARGET_COL, seq_len=SEQ_LEN, max_per_ticker=2)
test_examples  = build_examples(test_df,  feature_cols, TARGET_COL, seq_len=SEQ_LEN, max_per_ticker=2)

print("Train examples:", len(train_examples))
print("Val examples:", len(val_examples))
print("Test examples:", len(test_examples))

Train examples: 5000
Val examples: 1006
Test examples: 1006


## 7. Summarize Each Window For LLM Input

In [21]:
def summarize_window(window_df: pd.DataFrame, feature_cols, top_k_features=20):
    numeric = window_df[feature_cols].astype(float)

    # Keep only the first top_k_features for a cheap first prototype.
    use_cols = feature_cols[:top_k_features]
    numeric = numeric[use_cols]

    last_row = numeric.iloc[-1]
    mean_row = numeric.mean()
    std_row  = numeric.std().fillna(0.0)
    delta_row = numeric.iloc[-1] - numeric.iloc[0]

    summary = {
        "window_length": len(window_df),
        "last_date": str(window_df["Date"].iloc[-1].date()),
        "last_snapshot": {c: round(float(last_row[c]), 4) for c in use_cols},
        "window_mean":   {c: round(float(mean_row[c]), 4) for c in use_cols},
        "window_std":    {c: round(float(std_row[c]), 4) for c in use_cols},
        "window_delta":  {c: round(float(delta_row[c]), 4) for c in use_cols},
    }
    return summary

sample_summary = summarize_window(train_examples[0]["window_df"], feature_cols, top_k_features=12)
list(sample_summary.keys()), list(sample_summary["last_snapshot"].items())[:5]

(['window_length',
  'last_date',
  'last_snapshot',
  'window_mean',
  'window_std',
  'window_delta'],
 [('Open', -0.1971),
  ('High', -0.1929),
  ('Low', -0.1929),
  ('Close', -0.1898),
  ('Volume', -0.2157)])

## 8. Future Work - add news API

In [22]:
def get_news_context_stub(ticker: str, as_of_date: pd.Timestamp):
    # Replace this later with real headlines or summaries.
    # Keep blank for the first working version.
    return "No external news attached yet."

## 9. Output Structure for LLM Response

In [23]:
@dataclass
class StockLLMLabel:
    ticker: str
    as_of_date: str
    prediction: str          # "UP" or "DOWN"
    confidence: float        # 0.0 to 1.0
    short_reason: str

## 10. Convert Examples Into LLM-Ready Format
- Builds a global lookup of summarized stock examples for LLM use. Each example is indexed and contains ticker, date, engineered features, and optional news.
- This avoids recomputation, reduces token cost, and enables efficient, structured access when generating predictions or using tools.

In [24]:
# Store examples in a global dict so the tool can access them
example_store = {}

def refresh_example_store(examples, top_k_features=20):
    global example_store
    example_store = {}
    for i, ex in enumerate(examples):
        example_store[i] = {
            "ticker": ex["Ticker"],
            "as_of_date": str(pd.Timestamp(ex["Date"]).date()),
            "summary": summarize_window(ex["window_df"], feature_cols, top_k_features=top_k_features),
            "news_context": get_news_context_stub(ex["Ticker"], ex["Date"]),
        }

refresh_example_store(val_examples, top_k_features=20)
print("Training data size for LLM", len(example_store))

@function_tool
def get_stock_context(example_id: int) -> str:
    """Return stock context for a given example id as JSON."""
    if example_id not in example_store:
        return json.dumps({"error": f"example_id {example_id} not found"})
    return json.dumps(example_store[example_id], indent=2)

Training data size for LLM 1006


## 11. Agent Creation and Prompt

In [25]:
stock_agent = Agent(
    name="Stock Direction Research Agent",
    instructions=(
        "You are labeling stock examples for a machine learning dataset. "
        "Always call get_stock_context(example_id) before answering. "
        "Use ONLY the returned data. Do not assume any future information or outside knowledge.\n\n"

        "Your task is to predict whether the stock's next-day direction is UP or DOWN.\n\n"

        "Focus on these signals:\n"
        "- window_delta: overall trend across the window (positive suggests upward momentum, negative suggests downward momentum)\n"
        "- last_snapshot vs window_mean: whether the most recent values are above or below their recent average\n"
        "- window_std: volatility (higher volatility means weaker reliability and lower confidence)\n\n"

        "Decision rules:\n"
        "- Strong positive trend with supportive current values suggests UP\n"
        "- Strong negative trend with supportive current values suggests DOWN\n"
        "- If signals are mixed, weigh trend more than noise\n"
        "- If evidence is weak or conflicting, still choose UP or DOWN but lower confidence\n"
        "- Do not guess based on information not present in the context\n\n"

        "Return STRICT JSON with exactly these keys:\n"
        "ticker, as_of_date, prediction, confidence, short_reason\n\n"

        "Rules:\n"
        "- prediction must be exactly 'UP' or 'DOWN'\n"
        "- confidence must be a decimal float between 0 and 1\n"
        "- short_reason must be under 40 words\n"
        "- Do not include markdown, code fences, or any extra text outside the JSON"
    ),
    tools=[get_stock_context],
    model="gpt-4.1-mini"
)

## 12. Testing Agent on one example

In [26]:
test_prompt = (
    "Label example_id=0. "
    "You must call get_stock_context(example_id) exactly once before answering. "
    "Return only strict JSON."
)
result = await Runner.run(stock_agent, test_prompt)
print(result.final_output)

{
  "ticker": "A",
  "as_of_date": "2024-02-02",
  "prediction": "UP",
  "confidence": 0.7,
  "short_reason": "Moderate positive window delta on Close and RSI; last snapshot Close above window mean supports upward momentum despite mixed SMA and MACD signals."
}


## 13. Clean and Convert LLM Output
- This function cleans and converts LLM JSON output into a structured object so you can actually evaluate and use predictions.

In [27]:
def parse_agent_json(raw_text: str):
    try:
        raw_text = raw_text.strip()
        raw_text = raw_text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        data = json.loads(raw_text)

        return StockLLMLabel(
            ticker=data["ticker"],
            as_of_date=data["as_of_date"],
            prediction=data["prediction"],
            confidence=float(data["confidence"]),
            short_reason=data["short_reason"],
        )
    except Exception as e:
        print("Parse error:", e)
        print("Raw output:", raw_text)
        return None
parsed = parse_agent_json(result.final_output)

print('\n', parsed)


 StockLLMLabel(ticker='A', as_of_date='2024-02-02', prediction='UP', confidence=0.7, short_reason='Moderate positive window delta on Close and RSI; last snapshot Close above window mean supports upward momentum despite mixed SMA and MACD signals.')


## 14. Test LLM Using A Small Batch

In [28]:
async def label_examples_with_agent(examples, limit, top_k_features):
    refresh_example_store(examples, top_k_features=top_k_features)

    rows = []
    for i in range(min(limit, len(examples))):
        prompt = f"Label example_id={i}. Call the tool first. Output strict JSON only."
        try:
            result = await Runner.run(stock_agent, prompt)
            parsed = parse_agent_json(result.final_output)

            rows.append({
                "example_id": i,
                "Ticker": parsed.ticker,
                "Date": parsed.as_of_date,
                "llm_prediction": parsed.prediction,
                "llm_confidence": parsed.confidence,
                "llm_short_reason": parsed.short_reason,
                "true_target": int(examples[i]["target"]),
            })
        except Exception as e:
            rows.append({
                "example_id": i,
                "Ticker": examples[i]["Ticker"],
                "Date": str(pd.Timestamp(examples[i]["Date"]).date()),
                "llm_prediction": None,
                "llm_confidence": None,
                "llm_short_reason": f"ERROR: {e}",
                "true_target": int(examples[i]["target"]),
            })

    return pd.DataFrame(rows)

LLM_TRAIN_LIMIT = min(50, len(train_examples))
LLM_VAL_LIMIT   = min(50, len(val_examples))

llm_train_labels = await label_examples_with_agent(train_examples, limit=LLM_TRAIN_LIMIT, top_k_features=20)
llm_val_labels = await label_examples_with_agent(val_examples, limit=LLM_VAL_LIMIT, top_k_features=20)

print("Train LLM labels:", llm_train_labels.shape)
print("Val LLM labels:", llm_val_labels.shape)
llm_train_labels.head()


Train LLM labels: (50, 7)
Val LLM labels: (50, 7)


,example_id,Ticker,Date,llm_prediction,llm_confidence,llm_short_reason,true_target
0,0,A,2020-10-05,UP,0.80,Strong positive window_delta with recent value...,0
1,1,A,2020-10-06,UP,0.85,Strong positive window_delta trend and last_sn...,1
2,2,A,2020-10-07,UP,0.85,Strong positive window_delta trend with last s...,1
3,3,A,2020-10-08,UP,0.85,Strong positive window_delta trend with last s...,1
4,4,A,2020-10-09,UP,0.80,"Strong positive window_delta, last snapshot ab...",0


## 15. Evaluation LLM Predictions vs Real Values

In [29]:
eval_df = llm_val_labels.dropna(subset=["llm_prediction"]).copy()
eval_df["llm_pred_num"] = eval_df["llm_prediction"].map({"UP": 1, "DOWN": 0})

acc = accuracy_score(eval_df["true_target"], eval_df["llm_pred_num"])
print("LLM label accuracy on this batch:", round(acc, 4))
print(eval_df[["Ticker", "Date", "llm_prediction", "llm_confidence", "true_target"]].head(10))

LLM label accuracy on this batch: 0.5
  Ticker        Date llm_prediction  llm_confidence  true_target
0      A  2024-02-02             UP            0.70            0
1      A  2024-02-05             UP            0.70            1
2   AAPL  2024-02-02             UP            0.70            1
3   AAPL  2024-02-05             UP            0.70            1
4   ABBV  2024-02-02             UP            0.70            1
5   ABBV  2024-02-05             UP            0.75            1
6   ABNB  2024-02-02             UP            0.85            0
7   ABNB  2024-02-05             UP            0.80            0
8    ABT  2024-02-02             UP            0.70            0
9    ABT  2024-02-05             UP            0.55            1


## 16. Train A Student Model on The LLM Output

In [30]:
def flatten_summary_to_features(summary_dict):
    feats = {}
    for block_name in ["last_snapshot", "window_mean", "window_std", "window_delta"]:
        for k, v in summary_dict[block_name].items():
            feats[f"{block_name}__{k}"] = float(v)
    return feats

def build_student_frame(examples, llm_labels_df, top_k_features=20):
    rows = []
    label_map = llm_labels_df.set_index("example_id").to_dict("index")

    for i, ex in enumerate(examples):
        if i not in label_map:
            continue
        row = label_map[i]
        if row["llm_prediction"] not in ["UP", "DOWN"]:
            continue

        summary = summarize_window(ex["window_df"], feature_cols, top_k_features=top_k_features)
        feats = flatten_summary_to_features(summary)

        feats["llm_pred_num"] = 1 if row["llm_prediction"] == "UP" else 0
        feats["llm_confidence"] = float(row["llm_confidence"])
        feats["true_target"] = int(ex["target"])
        feats["Ticker"] = ex["Ticker"]
        feats["Date"] = str(pd.Timestamp(ex["Date"]).date())
        rows.append(feats)

    return pd.DataFrame(rows)

student_train_df = build_student_frame(train_examples, llm_train_labels, top_k_features=20)
student_val_df   = build_student_frame(val_examples,   llm_val_labels,   top_k_features=20)

In [31]:
feature_drop = ["Ticker", "Date", "true_target"]

X_train = student_train_df.drop(columns=feature_drop)
y_train = student_train_df["true_target"].astype(int)

X_val = student_val_df.drop(columns=feature_drop)
y_val = student_val_df["true_target"].astype(int)

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
print("Student model accuracy on validation:", round(accuracy_score(y_val, y_pred), 4))
print(classification_report(y_val, y_pred, zero_division=0))

Student model accuracy on validation: 0.6
              precision    recall  f1-score   support

           0       0.62      0.58      0.60        26
           1       0.58      0.62      0.60        24

    accuracy                           0.60        50
   macro avg       0.60      0.60      0.60        50
weighted avg       0.60      0.60      0.60        50



## 17. Save LLM labels
- Saves LLM results to reuse and not cost more money

In [32]:
os.makedirs("artifacts", exist_ok=True)

llm_train_labels.to_csv("artifacts/llm_train_labels.csv", index=False)
llm_val_labels.to_csv("artifacts/llm_val_labels.csv", index=False)

student_train_df.to_csv("artifacts/student_train_df.csv", index=False)
student_val_df.to_csv("artifacts/student_val_df.csv", index=False)

print("Saved:")
print("- artifacts/llm_train_labels.csv")
print("- artifacts/llm_val_labels.csv")
print("- artifacts/student_train_df.csv")
print("- artifacts/student_val_df.csv")

Saved:
- artifacts/llm_train_labels.csv
- artifacts/llm_val_labels.csv
- artifacts/student_train_df.csv
- artifacts/student_val_df.csv


## GRU Model
- Data Split
- Train model

In [33]:
class StockSequenceDataset(Dataset):
    def __init__(self, data: pd.DataFrame, feature_cols, target_col, seq_len=20):
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.seq_len = seq_len

        self.groups = []
        for ticker, g in data.groupby("Ticker", sort=False):
            g = g.reset_index(drop=True)
            if len(g) >= seq_len:
                self.groups.append(g)

        self.indices = []
        for gi, g in enumerate(self.groups):
            for end in range(seq_len - 1, len(g)):
                self.indices.append((gi, end))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        gi, end = self.indices[idx]
        g = self.groups[gi]

        start = end - self.seq_len + 1
        x = g.loc[start:end, self.feature_cols].to_numpy(dtype=np.float32)
        y = np.float32(g.loc[end, self.target_col])

        return torch.from_numpy(x), torch.tensor(y)

BATCH_SIZE = 256

train_dataset = StockSequenceDataset(train_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN)
val_dataset = StockSequenceDataset(val_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN)
test_dataset = StockSequenceDataset(test_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


print("Train sequences:", len(train_dataset))
print("Val sequences:", len(val_dataset))
print("Test sequences:", len(test_dataset))

class GRUClassifier(nn.Module):
    def __init__(self, num_features, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUClassifier(num_features=len(feature_cols)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


Train sequences: 424759
Val sequences: 83727
Test sequences: 82995


In [34]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device).float()

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / len(loader.dataset)
    acc = correct / total
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device).float()

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / len(loader.dataset)
    acc = correct / total
    return avg_loss, acc


best_val_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())
patience = 3
counter = 0
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_model_state)

test_loss, test_acc = eval_one_epoch(model, test_loader, criterion, device)
print(f"Final Test Loss: {test_loss:.4f} | Final Test Acc: {test_acc:.4f}")

Epoch 01 | Train Loss: 0.6916 | Train Acc: 0.5219 | Val Loss: 0.6928 | Val Acc: 0.5187
Epoch 02 | Train Loss: 0.6895 | Train Acc: 0.5329 | Val Loss: 0.6945 | Val Acc: 0.5132
Epoch 03 | Train Loss: 0.6867 | Train Acc: 0.5421 | Val Loss: 0.6950 | Val Acc: 0.5147
Epoch 04 | Train Loss: 0.6827 | Train Acc: 0.5534 | Val Loss: 0.7018 | Val Acc: 0.5008
Early stopping triggered.
Final Test Loss: 0.6952 | Final Test Acc: 0.5025


In [35]:
def get_sequence_model_predictions(model, examples, feature_cols, device):
    model.eval()
    rows = []

    with torch.no_grad():
        for i, ex in enumerate(examples):
            x = ex["window_df"][feature_cols].astype(float).values
            x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

            logit = model(x_tensor)
            prob = torch.sigmoid(logit).item()

            rows.append({
                "example_id": i,
                "Ticker": ex["Ticker"],
                "Date": str(pd.Timestamp(ex["Date"]).date()),
                "gru_prob": prob,
                "true_target": int(ex["target"]),
            })

    return pd.DataFrame(rows)

gru_train_preds = get_sequence_model_predictions(model, train_examples, feature_cols, device)
gru_val_preds = get_sequence_model_predictions(model, val_examples, feature_cols, device)

## Prepare LLM outputs
- This function cleans and converts LLM outputs into a consistent numeric format, and you avoid using test here to keep your final evaluation unbiased

In [36]:
def prep_llm_labels(llm_df):
    out = llm_df.copy()
    out = out[out["llm_prediction"].isin(["UP", "DOWN"])].copy()
    out["llm_pred_num"] = out["llm_prediction"].map({"UP": 1, "DOWN": 0})
    out["llm_confidence"] = pd.to_numeric(out["llm_confidence"], errors="coerce")
    out = out.dropna(subset=["llm_pred_num", "llm_confidence"])
    out["llm_confidence"] = out["llm_confidence"].clip(0.0, 1.0)
    return out[["example_id", "llm_pred_num", "llm_confidence"]]

llm_train_prepped = prep_llm_labels(llm_train_labels)
llm_val_prepped = prep_llm_labels(llm_val_labels)

## Merge GRU/LSTM + LLM outputs

In [37]:
ensemble_train_df = gru_train_preds.merge(llm_train_prepped, on="example_id", how="inner")
ensemble_val_df = gru_val_preds.merge(llm_val_prepped, on="example_id", how="inner")

ensemble_train_df.head()
ensemble_val_df.head()

,example_id,Ticker,Date,gru_prob,true_target,llm_pred_num,llm_confidence
0,0,A,2024-02-02,0.494010,0,1,0.7
1,1,A,2024-02-05,0.527445,1,1,0.7
2,2,AAPL,2024-02-02,0.457870,1,1,0.7
3,3,AAPL,2024-02-05,0.480230,1,1,0.7
4,4,ABBV,2024-02-02,0.534893,1,1,0.7


## Train the final ensemble model

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

ensemble_features = ["gru_prob", "llm_pred_num", "llm_confidence"]

X_train_ens = ensemble_train_df[ensemble_features]
y_train_ens = ensemble_train_df["true_target"].astype(int)

X_val_ens = ensemble_val_df[ensemble_features]
y_val_ens = ensemble_val_df["true_target"].astype(int)

ensemble_model = LogisticRegression(max_iter=1000, class_weight="balanced")
ensemble_model.fit(X_train_ens, y_train_ens)

val_pred = ensemble_model.predict(X_val_ens)
val_prob = ensemble_model.predict_proba(X_val_ens)[:, 1]

print("Ensemble Validation Accuracy:", round(accuracy_score(y_val_ens, val_pred), 4))
print("Ensemble Validation ROC AUC:", round(roc_auc_score(y_val_ens, val_prob), 4))
print(classification_report(y_val_ens, val_pred, zero_division=0))

Ensemble Validation Accuracy: 0.5
Ensemble Validation ROC AUC: 0.6026
              precision    recall  f1-score   support

           0       0.54      0.27      0.36        26
           1       0.49      0.75      0.59        24

    accuracy                           0.50        50
   macro avg       0.51      0.51      0.47        50
weighted avg       0.51      0.50      0.47        50

